# UCS532 P44 · COCO object hallucination audit

Choose **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. The notebook creates an isolated virtual environment on the Colab VM, runs the disjoint pilot and the 500-probe evaluation, and exports the result files. The upstream images and weights stay in the runtime cache. Keep the runtime connected until the download finishes.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

repo = Path("/content/cv-project")
remote = "https://github.com/Jaivesh8/cv-project.git"
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a Colab GPU under Runtime → Change runtime type.")
subprocess.run(["nvidia-smi"], check=True)
if not (repo / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", remote, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], check=True)
python = repo / ".venv/bin/python"
if not python.exists():
    subprocess.run(["python", "-m", "venv", str(repo / ".venv")], check=True)
subprocess.run([str(python), "-m", "pip", "install", "--no-cache-dir",
    "--index-url", "https://download.pytorch.org/whl/cu126",
    "torch==2.7.1+cu126", "torchvision==0.22.1+cu126"], check=True)
subprocess.run([str(python), "-m", "pip", "install", "--no-cache-dir",
    "-r", "requirements.txt"], cwd=repo, check=True)
subprocess.run([str(python), "-m", "pip", "check"], check=True)
subprocess.run([str(python), "-c", "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"], check=True)

In [ ]:
env = {**os.environ, "PYTHON": str(python), "MPLBACKEND": "Agg"}
args = ["--device", "cuda", "--batch-size", "8"]
subprocess.run(["bash", "run_all.sh", "--pilot", *args], cwd=repo, env=env, check=True)
subprocess.run(["bash", "run_all.sh", "--out", ".cache/colab-main", *args], cwd=repo, env=env, check=True)

In [ ]:
from google.colab import files
from zipfile import ZipFile, ZIP_DEFLATED

results = repo / ".cache/colab-main"
assert sum(1 for _ in (results / "predictions.jsonl").open()) == 6000
subprocess.run([str(python), "-m", "audit.gallery", "--results", str(results),
    "--out", "/content/p44_hallucination_gallery.png"], cwd=repo, env=env, check=True)
archive = "/content/P44_colab_results.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
    for artifact in sorted(results.iterdir()):
        if artifact.is_file():
            bundle.write(artifact, "results/" + artifact.name)
    bundle.write("/content/p44_hallucination_gallery.png", "hallucination_gallery.png")
print((results / "main.md").read_text())
files.download(archive)